# Classifier Signal Filter — Overlay on SMA Crossover

This notebook implements the **classifier overlay** from `technical_analysis/01_baseline_sma_crossover.ipynb`.

**Workflow**
1. Run the baseline SMA strategy in-sample to generate labeled trades.
2. Extract features at each entry bar (SMA spread, ATR%, momentum, RSI, time-of-day).
3. Walk-forward (5 folds): train a `GradientBoostingClassifier` on IS; apply as entry filter on OOS.
4. Compare filtered vs unfiltered out-of-sample performance per fold.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from source import (
    load_all, resample_ohlc,
    SMACrossoverStrategy, StrategyParams,
    Backtester, compute_metrics,
)

pd.set_option("display.float_format", "{:,.4f}".format)
plt.rcParams["figure.dpi"] = 110


In [ ]:
datasets = load_all(REPO_ROOT / "data")
forex_raw = {k: v for k, v in datasets.items() if v[0].group == "forex"}
b3_raw    = {k: v for k, v in datasets.items() if v[0].group == "b3"}

# Best timeframes from WFO in the baseline notebook: Forex 4h, B3 5min
TF_FOCUS = {"forex": "4h", "b3": "5min"}

def build_asset_tfs(raw_datasets, target_tfs):
    out = {}
    for tf in target_tfs:
        out[tf] = {}
        for (asset, src_tf), (meta, df_raw) in raw_datasets.items():
            out[tf][asset] = (
                resample_ohlc(df_raw, tf)
                if meta.timeframe.upper() in {"M1", "M5", "M15", "M30"}
                else df_raw
            )
    return out

asset_tfs = {
    "forex": build_asset_tfs(forex_raw, ["4h"]),
    "b3":    build_asset_tfs(b3_raw,    ["5min"]),
}

baseline_params = StrategyParams(fast=20, slow=50, atr_period=14,
                                  sl_atr_mult=2.0, tp_atr_mult=3.0)

print("Assets loaded per group/TF:")
for group, tfs in asset_tfs.items():
    for tf, assets in tfs.items():
        for asset, df in assets.items():
            print(f"  {group} {tf} {asset}: {len(df):,} bars")


In [ ]:
FEATURE_COLS = [
    "sma_spread_pct", "atr_pct",
    "momentum_5", "momentum_20",
    "rsi", "signal_dir",
    "hour_sin", "hour_cos",
]


def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, min_periods=period).mean()
    loss  = (-delta).clip(lower=0).ewm(alpha=1/period, min_periods=period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def build_features(sig_df):
    """Append ML features to a generate_signals() output DataFrame."""
    out = sig_df.copy()
    out["sma_spread_pct"] = (out["sma_fast"] - out["sma_slow"]) / out["close"]
    out["atr_pct"]        = out["atr"] / out["close"]
    out["momentum_5"]     = out["close"].pct_change(5)
    out["momentum_20"]    = out["close"].pct_change(20)
    out["rsi"]            = compute_rsi(out["close"])
    out["signal_dir"]     = out["signal"].astype(float)
    hour_rad = out.index.hour * 2 * np.pi / 24
    out["hour_sin"] = np.sin(hour_rad)
    out["hour_cos"] = np.cos(hour_rad)
    return out


def extract_trade_features(trades_df, features_df):
    """Return (X DataFrame, y array) aligned by trade entry time."""
    if trades_df.empty:
        return pd.DataFrame(columns=FEATURE_COLS), np.array([])
    entry_idx = pd.DatetimeIndex(trades_df["entry_time"].values)
    X = features_df.reindex(entry_idx)[FEATURE_COLS].fillna(0.0)
    y = (trades_df["pnl_points"].values > 0).astype(int)
    return X, y


class ClassifierFilteredStrategy:
    """Wraps SMACrossoverStrategy and suppresses signals the classifier rates as losses."""

    def __init__(self, base_strategy, clf):
        self.base   = base_strategy
        self.clf    = clf
        self.params = base_strategy.params

    def generate_signals(self, df):
        sig          = build_features(self.base.generate_signals(df))
        signal_mask  = sig["signal"] != 0
        if not signal_mask.any():
            return sig
        X = sig.loc[signal_mask, FEATURE_COLS].fillna(0.0)
        preds        = self.clf.predict(X)
        out          = sig.copy()
        reject_idx   = X.index[preds == 0]
        out.loc[reject_idx, "signal"] = 0
        return out


In [ ]:
N_SPLITS  = 5
OOS_RATIO = 0.25


def clf_walk_forward(df, base_strategy, n_splits=N_SPLITS, oos_ratio=OOS_RATIO):
    """Walk-forward classifier training and OOS evaluation."""
    n         = len(df)
    fold_size = n // n_splits
    rows              = []
    oos_trade_frames  = []
    bt                = Backtester(base_strategy)

    for i in range(n_splits):
        start = i * fold_size
        end   = start + fold_size if i < n_splits - 1 else n
        fold  = df.iloc[start:end]
        if len(fold) < 100:
            continue
        split  = max(1, int(len(fold) * (1 - oos_ratio)))
        is_df  = fold.iloc[:split]
        oos_df = fold.iloc[split:]
        if is_df.empty or oos_df.empty:
            continue

        # IS: run SMA → build feature/label matrix
        is_res = bt.run(is_df)
        is_sig = build_features(base_strategy.generate_signals(is_df))
        X_is, y_is = extract_trade_features(is_res.trades, is_sig)
        if len(X_is) < 10 or y_is.sum() < 3 or (y_is == 0).sum() < 3:
            continue

        # Train
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model",  GradientBoostingClassifier(
                n_estimators=100, max_depth=3,
                learning_rate=0.1, random_state=42,
            )),
        ])
        clf.fit(X_is, y_is)

        # OOS: baseline vs filtered
        oos_base     = bt.run(oos_df)
        oos_filtered = Backtester(ClassifierFilteredStrategy(base_strategy, clf)).run(oos_df)

        m_b = compute_metrics(oos_base)
        m_f = compute_metrics(oos_filtered)

        rows.append({
            "fold":         i,
            "oos_start":    oos_df.index[0],
            "oos_end":      oos_df.index[-1],
            "base_trades":  m_b["num_trades"],
            "filt_trades":  m_f["num_trades"],
            "base_sharpe":  m_b["sharpe_daily"],
            "filt_sharpe":  m_f["sharpe_daily"],
            "base_pnl":     m_b["total_pnl"],
            "filt_pnl":     m_f["total_pnl"],
            "base_wr":      m_b["win_rate"],
            "filt_wr":      m_f["win_rate"],
            "base_pf":      m_b["profit_factor"],
            "filt_pf":      m_f["profit_factor"],
            "is_accuracy":  float(clf.score(X_is, y_is)),
        })

        if not oos_filtered.trades.empty:
            t        = oos_filtered.trades.copy()
            t["fold"] = i
            oos_trade_frames.append(t)

    summary    = pd.DataFrame(rows)
    oos_trades = (
        pd.concat(oos_trade_frames, ignore_index=True)
        if oos_trade_frames else pd.DataFrame()
    )
    return summary, oos_trades


# Run for each group / asset
clf_results    = {}
clf_oos_trades = {}

for group, tf in TF_FOCUS.items():
    for asset, df in asset_tfs[group][tf].items():
        key = f"{group}/{asset}/{tf}"
        print(f"Walk-forward classifier — {key} ...", end="  ", flush=True)
        summary, oos = clf_walk_forward(df, SMACrossoverStrategy(baseline_params))
        clf_results[key]    = summary
        clf_oos_trades[key] = oos
        print(f"{len(oos)} filtered OOS trades")


In [ ]:
DISPLAY_COLS = [
    "fold", "oos_start", "oos_end",
    "base_trades", "filt_trades",
    "base_wr", "filt_wr",
    "base_sharpe", "filt_sharpe",
    "base_pnl", "filt_pnl",
    "is_accuracy",
]

for key, df_res in clf_results.items():
    print(f"\n{key}:")
    if df_res.empty:
        print("  no results")
        continue
    display(df_res[DISPLAY_COLS])


In [ ]:
keys = list(clf_results.keys())
n_rows = len(keys)
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 5 * max(n_rows, 1)))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for ri, key in enumerate(keys):
    summary = clf_results[key]
    if summary.empty:
        continue
    folds = summary["fold"].astype(str)
    x = np.arange(len(folds))
    w = 0.35

    ax = axes[ri, 0]
    ax.bar(x - w/2, summary["base_sharpe"].fillna(0), w, label="Baseline", color="steelblue")
    ax.bar(x + w/2, summary["filt_sharpe"].fillna(0), w, label="Filtered",  color="seagreen")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(folds)
    ax.set_title(f"{key} — OOS Sharpe per fold")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[ri, 1]
    ax.bar(x - w/2, summary["base_wr"].fillna(0), w, label="Baseline", color="steelblue")
    ax.bar(x + w/2, summary["filt_wr"].fillna(0),  w, label="Filtered",  color="seagreen")
    ax.axhline(0.5, color="gray", lw=0.8, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(folds)
    ax.set_title(f"{key} — OOS Win Rate per fold")
    ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

fig.suptitle("Classifier Filter: Baseline vs Filtered OOS Performance", fontsize=13)
fig.tight_layout()
plt.show()


## Takeaways

**What the classifier does**
- At each SMA crossover bar, it predicts whether the trade will be profitable
  using features known at bar close: SMA divergence, normalised ATR, momentum (5 and 20 bars),
  RSI, signal direction, and cyclically-encoded hour-of-day.
- Signals classified as likely losses are suppressed; all other mechanics (SL/TP, sizing) are unchanged.

**Reading the results**
- `filt_trades < base_trades` — the classifier is filtering some signals out.
- Higher `filt_wr` — filtered signals have a better win rate.
- `is_accuracy` near 0.50 — the classifier is barely beating random on IS data; expect little OOS lift.

**Known limitations and extensions**
- With a few hundred IS trades per fold, the classifier operates in a small-data regime —
  results will be noisy across seeds and assets.
- Richer features (volatility regime, spread breadth, order-flow imbalance) would provide
  a stronger IS signal.
- A probability threshold (`clf.predict_proba`) instead of a hard class boundary lets you
  tune the precision/recall trade-off.
- Try walk-forward hyperparameter tuning (e.g. `GridSearchCV` inside the IS fold) for
  a more honest OOS comparison.
